In [10]:
#Instalamos la libreria de STREAMLIT
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
#Instalamos la libreria de PLOTLY
%pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
#Para instalar npm en visual studio
#1.Desde Google escribir node.js
#2. Instalar la versión más recomendada
! npm install localtunnel


added 22 packages in 2s

3 packages are looking for funding
  run `npm fund` for details


In [13]:
%%writefile app.py 
#Creamos el archivo de la APP en el interprete principal (Phyton)
#####################################################
#Importamos librerias
import streamlit as st
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
######################################################
#Definimos la instancia
@st.cache_resource
######################################################
#Creamos la función de carga de datos
def load_data():
   #Lectura del archivo csv
   df=pd.read_csv("Ventas_limpio.csv")
    #Rellenamos nulos
   df =df.bfill()
   df =df.ffill()  
   #Creamos la lista de las variables categóricas sobre las que se generará la extracción de características(análisis univariado)
   Lista=['marca','nombre_cliente','estado','vendedor']
   return df, Lista
###############################################################################
#Cargo los datos obtenidos de la función "load_data"
df, Lista = load_data()
###############################################################################
#CREACIÓN DEL DASHBOARD
#Generamos las páginas que utilizaremos en el diseño
##############################################################################
#Generamos los encabezados para la barra lateral (sidebar)
st.sidebar.title("Ventas")

#Widget 1: Selectbox
#Menu desplegable de opciones de laa páginas seleccionadas
View= st.sidebar.selectbox(label= "Tipo de Análisis", options= ["Extracción de Características", 
                                              "Regresión Lineal", "Regresión No Lineal",
                                              "Regresión Logística", "ANOVA"])

 ####################################################################################
 
  ####################################################################################
 
 # CONTENIDO DE LA VISTA 5
if View == "ANOVA":
    #ANOVA 
    #Generamos la lista de variables numéricas
    numeric_df = df.select_dtypes(['float','int'])  #Devuelve Columnas
    Lista_num= ['precio_unitario','total_unidades','total_neto']                  #Devuelve lista de Columnas numéricas
    Lista_cat= ['marca','nombre_cliente','estado','vendedor']  #Devuelve lista de Columnas categóricas
    #Select box
    Variable_y= st.sidebar.selectbox(label= "Variable dependiente (y)", options= Lista_num)
    #Multi-select box
    Variable_x= st.sidebar.selectbox(label="Variable independiente (x)", options= Lista_cat)  


    ##########################################################################################
    Max_cat = int(df[Variable_x].nunique())
    N_cat = st.sidebar.slider(label="Número de categorías a mostrar",
                              min_value=0, max_value=Max_cat, value=min(10, Max_cat))
   
    ##########################################################################################
    Top_cat = df[Variable_x].value_counts().index[:N_cat]
    df_f = df[df[Variable_x].isin(Top_cat)]



    #Generamos los encabezados para el dashboard
    st.title("ANOVA")  
   

    ##########################################################################################
    if N_cat < 2:
        st.warning("Selecciona al menos 2 categorías para calcular el ANOVA.")
        st.stop()


    #Generamos el diseño del Layout deseado
    # Fila 1
    Contenedor_A, Contenedor_B = st.columns(2)
    #Redefinimos las variables 
    Var_num= df[Variable_y]
    Var_cat= df[Variable_x]  
    
    # Contenedor A: ANOVA y resultados
    with Contenedor_A: 
        st.subheader("Resultados del ANOVA")
        grupos = [grupo[Variable_y].dropna() for nombre, grupo in df_f.groupby(Variable_x)]
        f_valor, p_valor = stats.f_oneway(*grupos)

        st.write(f"**F-valor:** {f_valor:.3f}")
        st.write(f"**p-valor:** {p_valor:.4f}")
        if p_valor < 0.05:
            st.success("¡Hay diferencias significativas entre los grupos!")
        else:
            st.warning("No hay diferencias significativas entre los grupos.")

    # Contenedor B: Gráfico de barras
    with Contenedor_B:
        st.subheader("Gráfico de Barras: Media por grupo")

        resumen = df_f.groupby(Variable_x)[Variable_y].agg(['mean', 'sem']).reset_index()

        fig, ax = plt.subplots()

        # Dibujo de las barras manualmente con matplotlib para controlar yerr
        x = resumen[Variable_x].astype(str)  # Asegurar que son strings para el eje x
        y = resumen['mean']
        yerr = resumen['sem']

        ax.bar(x, y, yerr=yerr, capsize=5, color='skyblue', edgecolor='black')
        ax.set_ylabel(f"Media de {Variable_y}")
        ax.set_xlabel(Variable_x)
        ax.set_title("Media por grupo con Error Estándar")
        st.pyplot(fig)
 

Writing app.py
